<a href="https://colab.research.google.com/github/alee52/LLM_AgenticAI/blob/main/ensemble_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q --upgrade bitsandbytes trl

!wget -q https://raw.githubusercontent.com/alee52/LLM_AgenticAI/refs/heads/main/data_prep/evaluator.py -O util.py

In [ ]:
import os
import re
import math
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from datasets import load_dataset, Dataset, DatasetDict
from datetime import datetime
from peft import PeftModel

hf_token = userdata.get('HF_TOKEN')
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    login(hf_token, add_to_git_credential=True)
    print("HuggingFace token found and set as environment variable.")
else:
    print("HF_TOKEN not found in user data. Please ensure it is set in Colab secrets.")

from util import evaluate

In [ ]:
BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "categorize_products_no_cate"
HF_USER = "leearum95" # your HF name here!

LITE_MODE = False

DATA_USER = "leearum95"
DATASET_NAME = f"{DATA_USER}/items_prompts_full_no_category"
if LITE_MODE:
  # RUN_NAME = "2026-05-16_15.51.46-lite"
  REVISION = None
else:
  # RUN_NAME = "2026-05-08_18.44.24"
  RUN_NAME = "2026-05-16_15.51.46-lite"
  REVISION = None



PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"


# Hyper-parameters - QLoRA

QUANT_4_BIT = True
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

In [ ]:
groq_api_key = os.getenv("GROQ_API_KEY")
if groq_api_key:
    print("GROQ_API_KEY is set.")
else:
    print("GROQ_API_KEY is not set.")

openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
if openrouter_api_key:
    print("OPENROUTER_API_KEY is set.")
else:
    print("OPENROUTER_API_KEY is not set.")

hf_token = os.environ['HF_TOKEN']
if hf_token:
    print("HuggingFace token found.")
else:
    print("No HuggingFace token found.")

login(hf_token, add_to_git_credential=True)

#------------------------------

openrouter_url = "https://openrouter.ai/api/v1"